In [4]:
# Практика 3. Задание 1 – Сравнение моделей сентимент-анализа (без pandas/tabulate)

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

# ------------------------------------------------------------
# 1. Выбор моделей (первая заменена на работающую)
# ------------------------------------------------------------
# Модель A: rubert-tiny (3 класса: POSITIVE, NEUTRAL, NEGATIVE) – быстрая, лёгкая
model_name_a = "blanchefort/rubert-base-cased-sentiment"
tokenizer_a = AutoTokenizer.from_pretrained(model_name_a)
model_a = AutoModelForSequenceClassification.from_pretrained(model_name_a)
pipe_a = pipeline("sentiment-analysis", model=model_a, tokenizer=tokenizer_a)

# Модель B: rubert-base на Rusentiment (5 классов: positive, negative, neutral, skip, speech)
model_name_b = "seara/rubert-tiny2-russian-sentiment"
tokenizer_b = AutoTokenizer.from_pretrained(model_name_b)
model_b = AutoModelForSequenceClassification.from_pretrained(model_name_b)
pipe_b = pipeline("sentiment-analysis", model=model_b, tokenizer=tokenizer_b)

print("✓ Модели загружены:")
print(f"  1) {model_name_a} – метки: {pipe_a.model.config.id2label}")
print(f"  2) {model_name_b} – метки: {pipe_b.model.config.id2label}\n")

# ------------------------------------------------------------
# 2. Подготовка предложений и экспертных оценок
#    Экспертная оценка даётся в трёх классах (POSITIVE, NEUTRAL, NEGATIVE)
# ------------------------------------------------------------
sentences = [
    "Этот фильм – настоящий шедевр!",
    "Еда в ресторане была отвратительной.",
    "За окном идёт дождь, температура +5.",
    "Сегодня я получил долгожданную премию!",
    "Ну и обслуживание: ждали час, а принесли холодное.",
    "В целом неплохо, но могло быть и лучше.",
    "Книга читается на одном дыхании, очень увлекательно.",
    "Стул сломался, как только я на него сел.",
    "Прогноз погоды обещает солнце, но пока облачно.",
    "Это просто супер! Лучшее, что я пробовал!",
    "Никогда больше не куплю продукцию этого бренда.",
    "Скучный и бессмысленный фильм – разочарование.",
    "Сегодня сдал экзамен на отлично!",
    "Мне кажется, или этот чай невкусный?",
    "Прекрасный день для прогулки в парке.",
    "Ну конечно, опять начальник отличился, молодец просто!",
    "Отличная работа, сломали всё, что можно было сломать.",
    "Какой замечательный день, чтобы остаться дома и никуда не выходить.",
    "Очень вкусно, особенно если сравнивать с обычной едой.",
    "Ну просто супер, всё как я люблю – опять ничего не работает."
]

expert_labels = [
    "POSITIVE", "NEGATIVE", "NEUTRAL", "POSITIVE", "NEGATIVE",
    "NEUTRAL", "POSITIVE", "NEGATIVE", "NEUTRAL", "POSITIVE",
    "NEGATIVE", "NEGATIVE", "POSITIVE", "NEUTRAL", "POSITIVE",
    "NEGATIVE", "NEGATIVE", "NEUTRAL", "NEUTRAL", "NEGATIVE"
]

# ------------------------------------------------------------
# 3. Функция приведения меток модели B к трём классам
# ------------------------------------------------------------
def map_5class_to_3(label_5class):
    """Преобразует метки 5-классовой модели в 3 класса."""
    if label_5class in ("positive", "speech"):
        return "POSITIVE"
    if label_5class == "negative":
        return "NEGATIVE"
    return "NEUTRAL"  # neutral, skip

# ------------------------------------------------------------
# 4. Анализ и построение таблицы
# ------------------------------------------------------------
print("=" * 120)
print(f"{'№':<3} {'Предложение (первые 55 символов)':<55} {'Модель A':<18} {'Модель B':<18} {'Эксперт':<10} {'Совп.A':<7} {'Совп.B':<7}")
print("=" * 120)

correct_a = 0
correct_b = 0
total = len(sentences)

for i, (text, expert) in enumerate(zip(sentences, expert_labels), 1):
    # Предсказания модели A
    res_a = pipe_a(text)[0]
    label_a = res_a['label'].upper()        # уже POSITIVE/NEUTRAL/NEGATIVE
    score_a = res_a['score']

    # Предсказания модели B
    res_b = pipe_b(text)[0]
    label_b_raw = res_b['label'].lower()    # например 'positive'
    label_b_3 = map_5class_to_3(label_b_raw)
    score_b = res_b['score']

    # Сравнение с экспертом
    match_a = "Да" if label_a == expert else "Нет"
    match_b = "Да" if label_b_3 == expert else "Нет"

    if match_a == "Да":
        correct_a += 1
    if match_b == "Да":
        correct_b += 1

    # Обрезка длинного текста для вывода
    short_text = text[:52] + "..." if len(text) > 55 else text.ljust(55)

    print(f"{i:<3} {short_text:<55} "
          f"{label_a:<5}({score_a:.2f}){' ':<6} "
          f"{label_b_raw:<8}({score_b:.2f}){' ':<4} "
          f"{expert:<10} {match_a:<7} {match_b:<7}")

print("=" * 120)

# ------------------------------------------------------------
# 5. Итоговая точность
# ------------------------------------------------------------
print(f"\nИТОГОВАЯ ТОЧНОСТЬ:")
print(f"  Модель A ({model_name_a}): {correct_a}/{total} = {correct_a/total*100:.1f}%")
print(f"  Модель B ({model_name_b}): {correct_b}/{total} = {correct_b/total*100:.1f}%")

config.json:   0%|          | 0.00/943 [00:00<?, ?B/s]

d:\julia\Bachelor\4_course\2sem\Моделирование процессов в нотации BPMN\labs_bpmn\huggingface_labs\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jylij\.cache\huggingface\hub\models--blanchefort--rubert-base-cased-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/499 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/117M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: seara/rubert-tiny2-russian-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Модели загружены:
  1) blanchefort/rubert-base-cased-sentiment – метки: {0: 'NEUTRAL', 1: 'POSITIVE', 2: 'NEGATIVE'}
  2) seara/rubert-tiny2-russian-sentiment – метки: {0: 'neutral', 1: 'positive', 2: 'negative'}

№   Предложение (первые 55 символов)                        Модель A           Модель B           Эксперт    Совп.A  Совп.B 
1   Этот фильм – настоящий шедевр!                          NEUTRAL(0.80)       positive(0.93)     POSITIVE   Нет     Да     
2   Еда в ресторане была отвратительной.                    NEUTRAL(0.81)       negative(0.62)     NEGATIVE   Нет     Да     
3   За окном идёт дождь, температура +5.                    POSITIVE(0.52)       neutral (0.65)     NEUTRAL    Нет     Да     
4   Сегодня я получил долгожданную премию!                  POSITIVE(0.98)       positive(0.78)     POSITIVE   Да      Да     
5   Ну и обслуживание: ждали час, а принесли холодное.      NEGATIVE(0.75)       neutral (0.57)     NEGATIVE   Да      Нет    
6   В целом неплохо, но мо